# Fundamentos de Calidad de Datos (Junior)

Objetivos:
- Entender las dimensiones de calidad de datos.
- Medir y evaluar la calidad de un dataset.
- Identificar problemas comunes y estrategias de corrección.

## 1. ¿Qué es Calidad de Datos?

Calidad = grado en que los datos cumplen los requisitos para su uso previsto.

**Dimensiones clave** (DAMA-DMBOK):
- **Precisión (Accuracy):** Datos reflejan la realidad.
- **Completitud (Completeness):** Sin nulos inesperados.
- **Consistencia (Consistency):** Coherencia entre sistemas/fuentes.
- **Validez (Validity):** Cumplen reglas de negocio (formato, rango).
- **Unicidad (Uniqueness):** Sin duplicados no deseados.
- **Puntualidad (Timeliness):** Actualizados a tiempo.

## 2. Ejemplo: Dataset con Problemas de Calidad

In [ ]:
import pandas as pd
import numpy as np

# Dataset simulado con problemas
data = {
    'cliente_id': [1, 2, None, 4, 5, 5],
    'nombre': ['Ana', 'Luis', 'Carlos', 'Marta', 'Pedro', 'Pedro'],
    'email': ['ana@example.com', 'invalido', 'carlos@x.com', None, 'pedro@x.com', 'pedro@x.com'],
    'edad': [28, 200, 35, -5, 42, 42],
    'saldo': [1500.0, 2300.5, None, 800.0, 3200.0, 3200.0],
    'pais': ['CO', 'CO', 'PE', 'MX', 'AR', 'AR']
}
df = pd.DataFrame(data)
print(df)

## 3. Medición de Calidad por Dimensión

In [ ]:
# 1. Completitud: % de valores no nulos
completitud = (1 - df.isna().mean()) * 100
print('Completitud (%):\n', completitud.round(1))

# 2. Unicidad: identificar duplicados
duplicados = df.duplicated(subset=['cliente_id'], keep=False).sum()
print('\nFilas con cliente_id duplicado:', duplicados)

# 3. Validez: edad en rango razonable [0, 120]
edad_valida = df['edad'].between(0, 120).sum()
print('Edades válidas:', edad_valida, 'de', df['edad'].count())

# 4. Formato email: regex básico
import re
patron_email = r'^[\w\.-]+@[\w\.-]+\.\w+$'
df['email_valido'] = df['email'].apply(
    lambda x: bool(re.match(patron_email, str(x))) if pd.notna(x) else False
)
print('\nEmails válidos:', df['email_valido'].sum(), 'de', df['email'].notna().sum())

## 4. Reglas de Calidad como Assertions

In [ ]:
# Definir reglas y ejecutar
reglas = {
    'cliente_id_no_nulo': lambda d: d['cliente_id'].notna().all(),
    'edad_en_rango': lambda d: d['edad'].between(0, 120).all(),
    'sin_duplicados_cliente': lambda d: not d.duplicated(subset=['cliente_id'], keep=False).any(),
    'email_valido_si_presente': lambda d: (d['email'].notna() == d['email_valido']).all()
}

resultados = {nombre: regla(df) for nombre, regla in reglas.items()}
print('Resultados de calidad:')
for r, ok in resultados.items():
    print(f'  {r}: {"✓ OK" if ok else "✗ FALLO"}')

if not all(resultados.values()):
    print('\n⚠️ Dataset NO cumple con todas las reglas de calidad.')

## 5. Estrategias de Corrección

| Problema | Estrategia |
|----------|------------|
| Nulos | Imputar con media/moda/default o eliminar filas |
| Duplicados | Deduplicar con `drop_duplicates()` o lógica de merge |
| Valores fuera de rango | Filtrar o reemplazar (cap/floor) |
| Formato inválido | Validar en ingesta, rechazar o sanitizar |

In [ ]:
# Ejemplo de corrección
df_limpio = df.copy()
# Eliminar filas con cliente_id nulo
df_limpio = df_limpio[df_limpio['cliente_id'].notna()]
# Deduplicar por cliente_id (mantener primera)
df_limpio = df_limpio.drop_duplicates(subset=['cliente_id'], keep='first')
# Cap edad a [0, 120]
df_limpio['edad'] = df_limpio['edad'].clip(0, 120)
# Llenar saldo nulo con 0
df_limpio['saldo'].fillna(0, inplace=True)

print('Dataset limpio:\n', df_limpio)
print('\nFilas originales:', len(df), '| Después de limpieza:', len(df_limpio))

## 6. Ejercicio

Dado el dataset `ventas_raw` (inventado o cargar de CSV):
1. Calcula el % de completitud por columna.
2. Identifica duplicados por `pedido_id`.
3. Valida que `monto` sea positivo.
4. Reporta un resumen de calidad con métricas clave.

In [ ]:
# Plantilla para ejercicio
# ventas_raw = pd.read_csv('ventas.csv')
# completitud = ...
# duplicados = ...
# monto_valido = ...
print('Solución del ejercicio aquí')

## 7. Resumen

- Calidad de datos es multidimensional (precisión, completitud, consistencia, validez, unicidad, puntualidad).
- Medir calidad con métricas cuantitativas.
- Implementar reglas como assertions ejecutables.
- Estrategias de corrección: imputación, deduplicación, filtrado.